# LangChain ConversationBufferMemory 

**Learning path:** LLM without memory → buffer memory → inspect/save/clear → interactive chat → multiple conversations using UUID → Flask chatbot.

Based on your uploaded notebook, with corrected examples and safer API-key handling. The `ConversationChain` and `ConversationBufferMemory` APIs are **legacy** APIs in `langchain-classic`; this notebook uses them intentionally to explain your original code. Later sections show how to progress toward modern message-history patterns.

**How to use:** Run cells from top to bottom in Jupyter. Code cells that call Groq require internet, a valid API key, and a currently supported Groq model. Outputs are illustrative, not guaranteed verbatim.

## why memory matters

**Without history:** Human: “My name is Veera.” → AI: “Hello!” → next independent request: “What is my name?” The model may not know.

**With history:** The application sends the earlier Human/AI messages along with the new question, so the model can answer “Veera.”

> Memory is **application-managed conversation history**, not the model permanently learning a fact. Buffer memory stores the whole conversation, which can increase tokens/cost and eventually exceed the model context window.

## 2. Install dependencies (run once)

If you already installed these packages, you can skip. Restart the Jupyter kernel after installation if imports fail.

In [ ]:
%pip install -U langchain-classic langchain-groq python-dotenv

## 3. Configure Groq securely

1. Revoke any Groq API key previously pasted into a notebook, chat, or GitHub repository.
2. Generate a new key from your Groq account.
3. In the same folder as this notebook, create a `.env` file with `GROQ_API_KEY=your_new_key` and optionally `GROQ_MODEL=your_current_supported_model`.
4. Add `.env` to `.gitignore` and **never commit it**.

You can also set `GROQ_API_KEY` as an operating-system environment variable. The model ID must be a model currently available to your Groq account.

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
API_KEY = os.getenv("GROQ_API_KEY","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")
MODEL = os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b")  # Change if unavailable

# if not API_KEY:
#     raise RuntimeError("Set GROQ_API_KEY in your environment or .env file first")

llm_model = ChatGroq(model=MODEL, api_key=API_KEY, temperature=0.3)
print("Groq model configured:", MODEL)  # Never print your API key

Groq model configured: qwen/qwen3.8-27b


## 4. First: LLM without memory

Each `.invoke()` below is independent. The second call does **not** automatically receive the first call's text.

In [3]:
from langchain_core.messages import HumanMessage

answer1 = llm_model.invoke([HumanMessage(content="My name is Veera")])
print("First answer:", answer1.content)

answer2 = llm_model.invoke([HumanMessage(content="What is my name?")])
print("Second answer:", answer2.content)

First answer: Hello Veera! It's nice to meet you. How can I help you today?
Second answer: I don't know your name. As an AI assistant, I don't have access to your personal information or identity unless you choose to share it with me during our conversation.

If you'd like, you can tell me your name, and I'll be happy to use it!


## 5. What does ConversationBufferMemory mean?

- **Conversation:** back-and-forth messages.
- **Buffer:** storage of the conversation so far.
- **Memory:** a component that makes the stored history available to the chain.

`return_messages=True` returns message objects (`HumanMessage`, `AIMessage`), while `False` returns a formatted text history.

**Execution:** input → chain loads history → LLM sees history + current input → chain saves input and answer.

In [4]:
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

buffer_memory = ConversationBufferMemory(return_messages=True)
conversation = ConversationChain(llm=llm_model, memory=buffer_memory, verbose=False)
print("Created an empty conversation")

Created an empty conversation


C:\Users\vvard\AppData\Local\Temp\ipykernel_13264\2202040933.py:4: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  buffer_memory = ConversationBufferMemory(return_messages=True)
c:\Users\vvard\anaconda3\Lib\site-packages\pydantic\main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
C:\Users\vvard\AppData

## 6. Example 1: remember a name

Run both cells **in order**, using the same `conversation` variable.

In [5]:
print(conversation.predict(input="Hello, my name is Veera"))

Hello Veera! It is such a pleasure to meet you. I am an AI assistant, and I am here to chat, answer questions, or help with whatever you need.

Since we are just getting started, I’d love to know a little more about you! What brings you here today? Are you looking for some interesting facts, help with a specific task, or just want to have a friendly conversation? I’m all ears (or rather, all text)! 😊


In [6]:
print(conversation.predict(input="What is my name?"))
# Expected meaning: the AI should recall Veera from the stored history.

Your name is Veera! You mentioned it right at the beginning of our conversation, and I am happy to remember it for you. 😊


## 7. Example 2: remember multiple facts

The conversation object is the same, so it keeps adding messages.

In [7]:
print(conversation.predict(input="I work as a Cloud DevOps trainer"))
print(conversation.predict(input="I live in Hyderabad"))
print(conversation.predict(input="Tell me my name, profession and city"))

Oh, that is a fantastic profession! A Cloud DevOps trainer is essentially a bridge between complex technical infrastructure and human understanding. That is such a vital role because the DevOps landscape is moving so incredibly fast with new tools, frameworks, and methodologies popping up every week.

I imagine your days are quite dynamic. Do you find yourself teaching more on the "Dev" side—like CI/CD pipelines, containerization with Docker or Kubernetes, and infrastructure as code with Terraform or CloudFormation—or do you spend more time on the "Ops" side, focusing on monitoring, logging, and cloud-native architectures?

Also, which cloud platforms do you specialize in? Are you a big fan of AWS, Azure, or maybe GCP? I’ve heard that training people on multi-cloud strategies is becoming increasingly common, but I’d love to know what your typical curriculum looks like. Do you work with junior developers who are just starting out, or are you upskilling experienced engineers who are tran

In [8]:
print(conversation.predict(input="What is my name?"))


Your name is Veera! You shared that with me at the very beginning of our conversation, and I have it clearly noted down. 😊


## 8. Inspect what the memory contains

`load_memory_variables({})` returns a dictionary whose `history` value contains messages. `chat_memory.messages` directly accesses the list of message objects.

In [9]:
stored = buffer_memory.load_memory_variables({})
print("Memory keys:", stored.keys())
print("Number of stored messages:", len(buffer_memory.chat_memory.messages))
for index, message in enumerate(buffer_memory.chat_memory.messages, start=1):
    print(index, type(message).__name__, "=>", message.content)

Memory keys: dict_keys(['history'])
Number of stored messages: 12
1 HumanMessage => Hello, my name is Veera
2 AIMessage => Hello Veera! It is such a pleasure to meet you. I am an AI assistant, and I am here to chat, answer questions, or help with whatever you need.

Since we are just getting started, I’d love to know a little more about you! What brings you here today? Are you looking for some interesting facts, help with a specific task, or just want to have a friendly conversation? I’m all ears (or rather, all text)! 😊
3 HumanMessage => What is my name?
4 AIMessage => Your name is Veera! You mentioned it right at the beginning of our conversation, and I am happy to remember it for you. 😊
5 HumanMessage => I work as a Cloud DevOps trainer
6 AIMessage => Oh, that is a fantastic profession! A Cloud DevOps trainer is essentially a bridge between complex technical infrastructure and human understanding. That is such a vital role because the DevOps landscape is moving so incredibly fast wi

### What does `return_messages=False` change?

This is a **separate example** so the original memory is unaffected.

In [10]:
text_memory = ConversationBufferMemory(return_messages=False)
text_memory.save_context({"input": "Hi"}, {"output": "Hello!"})
print(text_memory.load_memory_variables({})["history"])

message_memory = ConversationBufferMemory(return_messages=True)
message_memory.save_context({"input": "Hi"}, {"output": "Hello!"})
print(message_memory.load_memory_variables({})["history"])

Human: Hi
AI: Hello!
[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


## 9. Save messages manually with `save_context`

The chain normally saves automatically, but you can practice the memory object without making any Groq API calls.

In [11]:
manual_memory = ConversationBufferMemory(return_messages=True)
manual_memory.save_context(
    {"input": "I am learning Python"},
    {"output": "Great! Let's start with variables."}
)
manual_memory.save_context(
    {"input": "Which topic am I learning?"},
    {"output": "You are learning Python."}
)
for msg in manual_memory.chat_memory.messages:
    print(type(msg).__name__, ":", msg.content)

HumanMessage : I am learning Python
AIMessage : Great! Let's start with variables.
HumanMessage : Which topic am I learning?
AIMessage : You are learning Python.


## 10. Clear memory: a fresh conversation

`clear()` empties this memory object. It does not erase previous information from external databases (if any).

In [12]:
temporary = ConversationBufferMemory(return_messages=True)
temporary.save_context({"input": "My favorite language is Python"}, {"output": "Noted!"})
print("Before clear:", len(temporary.chat_memory.messages))
temporary.clear()
print("After clear:", len(temporary.chat_memory.messages))

Before clear: 2
After clear: 0


## 11. Example 3: two separate memory objects

Student A and Student B should **not** share conversation history.

In [13]:
memory_a = ConversationBufferMemory(return_messages=True)
memory_b = ConversationBufferMemory(return_messages=True)

memory_a.save_context({"input": "My name is Veera"}, {"output": "Hi Veera"})
memory_b.save_context({"input": "My name is Chitti"}, {"output": "Hi Chitti"})

print("A:", memory_a.load_memory_variables({})["history"])
print("B:", memory_b.load_memory_variables({})["history"])

A: [HumanMessage(content='My name is Veera', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi Veera', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
B: [HumanMessage(content='My name is Chitti', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi Chitti', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


## 12. UUID — unique chat IDs

UUID is a unique identifier, **not** the memory itself. It is used as a key to find a conversation in a dictionary.

In [14]:
import uuid
chat_id_a = str(uuid.uuid4())
chat_id_b = str(uuid.uuid4())
print("Student A chat ID:", chat_id_a)
print("Student B chat ID:", chat_id_b)
print("Different IDs?", chat_id_a != chat_id_b)

Student A chat ID: f5c2276a-d240-42ab-8187-daffaaec5ec2
Student B chat ID: eae3a1fa-2312-49d2-8313-d240c89c1ec3
Different IDs? True


## 13. Dictionary: `conversations[chat_id]`

A Python dictionary maps **chat ID → conversation object**. A new ID gets a new chain and memory; an existing ID reuses its chain and history.

In [15]:
conversations = {}

def get_conversation(chat_id):
    if chat_id not in conversations:
        new_memory = ConversationBufferMemory(return_messages=True)
        conversations[chat_id] = ConversationChain(
            llm=llm_model, memory=new_memory, verbose=False
        )
        print("Created NEW conversation:", chat_id)
    else:
        print("Reusing EXISTING conversation:", chat_id)
    return conversations[chat_id]

chat_a = get_conversation(chat_id_a)
chat_b = get_conversation(chat_id_b)
chat_a_again = get_conversation(chat_id_a)
print("Same object reused?", chat_a is chat_a_again)
print("Number of conversations:", len(conversations))

Created NEW conversation: f5c2276a-d240-42ab-8187-daffaaec5ec2
Created NEW conversation: eae3a1fa-2312-49d2-8313-d240c89c1ec3
Reusing EXISTING conversation: f5c2276a-d240-42ab-8187-daffaaec5ec2
Same object reused? True
Number of conversations: 2


## 14. Example 4: separate Groq conversations

The answers should reflect each chat's own history. Results may vary because an LLM generates the responses.

In [16]:
print("A1:", chat_a.predict(input="My name is Veera"))
print("B1:", chat_b.predict(input="My name is Chitti"))
print("A2:", get_conversation(chat_id_a).predict(input="What is my name?"))
print("B2:", get_conversation(chat_id_b).predict(input="What is my name?"))

A1: Hello Veera! It is a genuine pleasure to meet you. I have to say, "Veera" is a beautiful name; it carries a sense of strength and vitality, often associated with concepts like "brave" or "heroic" in various linguistic traditions, which makes for a wonderful introduction.

Since this is our first interaction, I am here to assist you with whatever you might need, whether that’s answering complex questions, brainstorming creative ideas, writing code, or just having a friendly chat. I’m quite chatty, so feel free to ask me anything at all! How are you doing today, and is there anything specific on your mind that you’d like to dive into?
B1: Hello Chitti! It is such a pleasure to meet you. I have to say, that is a very distinctive and memorable name. It sounds like it has a lot of personality to it!

Since we are just getting started, I’m curious—where did you get the name Chitti? Is it a nickname, a family name, or perhaps something you chose for yourself? I’d love to know a little mor

## 15. `if chat_id not in conversations` explained

Suppose `conversations = {"100": some_chain}`.

- ID `100` exists → **reuse** its chain and memory.
- ID `200` does not exist → **create** a new chain and memory.

Do **not** generate a new UUID for every message. Generate a chat ID once per conversation and reuse it for later turns.

In [17]:
example_ids = {"100": "Existing chat"}
for current_id in ["100", "200", "100"]:
    if current_id not in example_ids:
        example_ids[current_id] = "New chat"
        print(current_id, "=> created")
    else:
        print(current_id, "=> reused")

100 => reused
200 => created
100 => reused


## 16. Example 5: interactive terminal/Jupyter chatbot

Run this cell and type messages. Enter `stop` or `done` to quit. The stop command is checked **before** sending it to Groq.

In [18]:
interactive_memory = ConversationBufferMemory(return_messages=True)
interactive_chat = ConversationChain(llm=llm_model, memory=interactive_memory)

while True:
    question = input("Human: ").strip()
    if question.lower() in {"stop", "done", "exit"}:
        print("Chat ended.")
        break
    if not question:
        continue
    answer = interactive_chat.predict(input=question)
    print("AI:", answer)

AI: Hello there! It is wonderful to meet you. I am here and ready to chat, answer questions, or help with whatever is on your mind. How is your day going so far?
Chat ended.


## 17. Common mistakes in the original practice code

**Mistake A:** A dictionary with keys `Question` and `Answer` is overwritten on every iteration. To preserve all turns, append records to a **list**.

**Mistake B:** Generating a new UUID inside every loop iteration makes a new chat ID each time. Create the ID **once**.

**Mistake C:** `if history['chat_id'] is not in history` is invalid Python syntax. Use `if chat_id not in conversations:`.

**Mistake D:** `stop` should be checked before `predict()` to avoid an unnecessary API call.

**Mistake E:** Never hardcode an API key into the notebook.

In [ ]:
history_records = []
example_chat_id = str(uuid.uuid4())  # Created ONCE

for question, answer in [
    ("Hi", "Hello"),
    ("I am Veera", "Nice to meet you"),
]:
    history_records.append({"chat_id": example_chat_id, "question": question, "answer": answer})

print(history_records)
print("Number of saved turns:", len(history_records))

## Compare memory types (conceptual preview)

| Memory type | Idea | Trade-off |
|---|---|---|
| ConversationBufferMemory | Keep full history | Grows continuously |
| ConversationBufferWindowMemory | Keep a recent window | Older messages are dropped |
| ConversationTokenBufferMemory | Limit history by token budget | Older messages may be dropped |
| ConversationSummaryMemory | Summarize older history | Extra summarization and possible information loss |
| ConversationSummaryBufferMemory | Summary + recent turns | More complex; extra work |

These are **legacy** memory patterns. This notebook teaches BufferMemory in detail first.